In this notebook, the main comparison of the search algorithm is run using a PC-KNN, a learned metric, and the energy of the logits on a fixed budget of 60 evaluations. For this, we search for the hyperparameters under this specific budget restraint. The hyperparameters are then evaluated on a test set that was also transformed. This accuracy is averaged over several runs, with the runs chosen based on the size of the validation set. The mean and SE are reported over the test set. Note: We only do 1 run of finding hyperparameters.

As a second metric we calculate the worst and best energy across all runs and algorithms. We use this to calculte the distance to the worst and divide by the range to get ranges in 0,1. The result shows how well a search performs relative to others.
This score is averaged to get a score indicating how far we get to the best result any optimizer finds. As we test a lot of optimizer over several runs this is likely close to the best possible value.

Dataset can be changed to the values 'bigger_mnist', 'bigger_emnist', 'tu_berlin', and 'modelnet10'.


In [ ]:
#technical the per class knn is set to 1 neighbor which is not the default used for the search in 3_1.(Done this earlier generally slighly larger values are better)

In [ ]:


import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_mnist"  #choose one of the below

default_architecutre_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",

}

architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:

from data.get_dataset import get_dataset_info
from data.get_dataset import get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info, path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x = next(iter(test_loader_transformed))[0]

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]


In [ ]:


from src.utils.eval.vis import vis_dataset

vis_dataset(train_loader, val_loader, test_loader_transformed)

In [ ]:

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparision_over_budget")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
from model.get_model import get_network
from model.train import train_and_get_model

model = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train = f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model, model_dir_path, modelname, train_loader, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)



In [ ]:
model.eval().to(device)
from model.basic_networks import make_deterministic

make_deterministic(model)

In [ ]:
from src.utils.eval.main_model import evaluate_base_model

print("Base Model on transformed test data")
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)

In [ ]:
print("Base Model on test data")
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
print("Base Model on val data")
res = evaluate_base_model(model, val_loader, device)
print(res)

In [ ]:
#chek if data is image data, used for energy model.
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:
#get transfomations for search.
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

In [ ]:
# get last layer
from model.get_model import get_network_layer

layer, layer_io = get_network_layer(dataset_info, architecture, 0, num_classes=None, num_rotations=8)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence

logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)
problem_energy_logits = TransformationProblem(logit_energy, transform_seq, consolidate_method="consolidate_simple")
#test ot
from search.random_search import RSLR

random_search = RSLR(initial_samples=120, local_max_steps=0)

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search, ITSWRAPPER


In [ ]:
#Energy Baseline
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("energy_confidence_transformed"), show_progress=True,
    repeats=1)

In [ ]:
model.to(device).eval()

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
#create energy model
from model.train import train_or_load_energy_model
from model.tu_berlin_architectures import StrokeAugment


# dec strat tells which samples are considered positive and negative for energy model.
# Here it was better to simply set it so that correct classifications by the base model are positive the other are negative class instead of using whether they were transformed or not. Likely due to the latter resulting in very narrow regions around the data that a search has difficulty finding.
def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler

energy_model2 = get_network(dataset_info, architecture, num_classes=1).to(device)

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = src.utils.augments.build_default_augmentations()
else:
    if dataset in ["tu_berlin"]:
        transform_true_function = None
        affine_augment = StrokeAugment()
    else:
        transform_true_function = None
        affine_augment = None

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)

energy_conf2 = train_or_load_energy_model(
    energy_model2, model_dir_path, f"{modelname}_energy2", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)


In [ ]:
model.to(device).eval()

In [ ]:

make_deterministic(energy_model2)

In [ ]:
energy_conf2.to(device).eval()
problem_energy2 = TransformationProblem(energy_conf2, transform_seq, consolidate_method="consolidate_simple")

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_energy2,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_transformed"), show_progress=True,
    repeats=1)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:

from embedding_cache import LayerEmbeddingCache

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

cache_train = LayerEmbeddingCache(model, train_loader_no_shuffle,
                                  cache_dir=os.path.join(embedding_cache_path, cache_name_train))

dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)
embeddings_t, final_t, classes_t = cache_train.__call__(layer, capture_modes=layer_io, flatten=True)

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import PerClassKNNConfidence

nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None, computation_mode="masked", k=1)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.to(device)

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple")
model.eval().to(device)

In [ ]:
#benchmark model and dual output model

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("knn_per_class_confidence_transformed"), show_progress=True,
    repeats=1, overwrite=False)

In [ ]:
x = next(iter(train_loader_no_shuffle))
res1 = dual_output_model(x[0].to(device))[0].cpu().detach().numpy()
res2 = embeddings_t[:x[0].shape[0]].cpu().detach().numpy()
if not np.allclose(res1, res2):
    pass
else:
    print("Model is deterministic.")

del res1
del res2
del x

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#main cells that prepares problems. Disables grad to models and prepares problem for specific requirements some seraches have then performs the search.
from src.utils.replacer import replace_rotation_transforms, replace_rotation_transforms_2vec
import os
import optuna
import torch
import gc

from hyper_param.search.objective_generators import (
    make_search_objective,
    save_best_trial_params,
    build_search_algorithm,
    _cost_shgo,
)
from hyper_param.search.default_values import load_params, get_default_params, save_params
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

model.eval().to(device)
#detach model for efficent gradients
for param in model.parameters():
    param.requires_grad = False

for param in energy_model2.parameters():
    param.requires_grad = False

# Algorithms and budgets
all_algos = ["shgo", "parallel_sa",
             "pso", "cd", "wcd",
             "pgd", "its", "random_search", "cd_multi_cyclus", "wcd_lattice"
             ]
#NOTE: shgo is random search. It previosly followed shgo more closely but it had to much overhead so replaced by a local minimzer or simple topk selection processs for local points. Should be renamed but would invalidate the stored values.
# Add complex/quaternion variants
complex_algos = ["shgo_c", "pgd_c"]
all_algos.extend(complex_algos)

if dataset == "modelnet10":
    #all_algos.append("shgo_individual")
    all_algos.append("wcd_lat_ind")
    all_algos.remove("wcd_lattice")
    all_algos.append("pgd_vector")
    all_algos.append("shgo_vector")
    #all_algos.append("random_search_vector") #had similar results to normal random search

if dataset in ["tu_berlin"]:
    all_algos = [
        "shgo", "parallel_sa",
        "pso", "cd", "wcd", "its", "random_search", "wcd_lattice", "cd_multi_cyclus"]
    grad_weight = 99999999
else:
    grad_weight = 2

budgets = [60]

grad_weight_algos = {"shgo", "shgo_individual", "pgd",
                     "shgo_c", "pgd_c", "shgo_c_3", "pgd_c_3", "pgd_vector", "shgo_vector"}
default_trials = 30
eval_repeats = 5
if dataset in ["coil100", "tu_berlin", "modelnet10"]:
    eval_repeats = 10

show_progress = True

# Problem configurations
problems = [problem_energy_logits, problem_nn_pytorch_pretrained, problem_energy2]
problem_names = ["logit_energy", "knn_per_class", "learned_energy"]

# Create complex/quaternion versions of problems
problems_complex = [replace_rotation_transforms(p) for p in problems]
problem_names_complex = problem_names

# New problems for shgo_individual
problems_individual = [ITSWRAPPER._prepare_problem(p) for p in problems]
problem_names_individual = problem_names

problems_rotvec = [replace_rotation_transforms_2vec(p) for p in problems]
problem_names_rotvec = problem_names

for p in problems:
    p.max_batch_size = dataset_info.batch_size_search
for p in problems_individual:
    p.max_batch_size = dataset_info.batch_size_search
for p in problems_complex:
    p.max_batch_size = dataset_info.batch_size_search

# Base results directory
base_results_dir = os.path.join(
    current_path, "experiment_files", "search_results",
    str(dataset), dataset_info.transform_seq_name, str(architecture)
)
os.makedirs(base_results_dir, exist_ok=True)

assert len(problems) == len(problem_names), "Mismatch between problems and problem_names."

# Mapping from complex variants to their base algorithms
algo_variant_mapping = {
    "pgd_c": "pgd",
    "shgo_c": "shgo",
    "shgo_c_3": "shgo",
    "pgd_c_3": "pgd",
    "pgd_vector": "pgd",
    "shgo_vector": "shgo",
    "random_search_vector": "random_search",
}
grad_weight_orig = grad_weight
for budget in budgets:
    print(f"\n=== Budget: {budget} ===")
    for algo in all_algos:
        gc.collect()
        torch.cuda.empty_cache()
        print(f"\n--- Algorithm: {algo} ---")

        if algo == "shgo_c_3" or algo == "pgd_c_3":
            grad_weight = 3
        else:
            grad_weight = grad_weight_orig

        # Determine which problems to use
        if algo in complex_algos:
            current_problems = problems_complex
            current_problem_names = problem_names_complex
        elif algo in ["shgo_individual", "wcd_lat_ind", "its"]:
            current_problems = problems_individual
            current_problem_names = problem_names_individual
        elif algo in ["pgd_vector", "shgo_vector", "random_search_vector"]:
            current_problems = problems_rotvec
            current_problem_names = problem_names_rotvec
        else:
            current_problems = problems
            current_problem_names = problem_names

        for p in current_problems:
            p.max_batch_size = dataset_info.batch_size_search

        # Map algorithm names for construction
        if algo in complex_algos:
            algo_name_for_path = algo_variant_mapping[algo]
        elif algo == "shgo_individual":
            algo_name_for_path = "shgo"
        elif algo == "wcd_lat_ind":
            algo_name_for_path = "wcd_lattice"
        elif algo in ["pgd_vector", "shgo_vector", "random_search_vector"]:
            algo_name_for_path = algo_variant_mapping[algo]
        else:
            algo_name_for_path = algo

        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        os.makedirs(algo_dir, exist_ok=True)

        param_path = os.path.join(algo_dir, "best.yml")
        print(f"Result directory: {algo_dir}")

        # Load stored params or optimize
        stored_params = load_params(param_path) if os.path.exists(param_path) else None
        if stored_params is None and (algo != "cd" and "random_search" not in algo):
            default_params_kwargs = {}
            if algo in grad_weight_algos:
                default_params_kwargs["grad_weight"] = grad_weight
            default_params = get_default_params(algo_name_for_path, budget, **default_params_kwargs)
            print("Default params (config):", default_params)

            objective_kwargs = {}
            if algo in grad_weight_algos:
                objective_kwargs["grad_weight"] = grad_weight

            objective = make_search_objective(
                algo=algo_name_for_path,
                model=model,
                val_loader=val_loader_transformed,
                problem=current_problems,
                budget=budget,
                device=str(device),
                repeats=1,
                **objective_kwargs,
            )

            pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1)
            study = optuna.create_study(direction="maximize", pruner=pruner)
            study.enqueue_trial(default_params)

            study.optimize(objective, n_trials=default_trials, show_progress_bar=False)

            print(f"[{algo}] Best validation value:", study.best_value)
            print("Suggested params:", study.best_trial.params)
            print("Full params:", study.best_trial.user_attrs.get("full_params"))

            save_best_trial_params(study, algo=algo_name_for_path, path=param_path)
            stored_params = load_params(param_path)
            print("Saved best params to:", param_path)
        else:
            print("Found stored best params, skipping optimization.")
            if "cd" == algo:
                print("Note: 'cd' algorithm requires no hyperparameter optimization.")
                stored_params = {}

        # SHGO-only sanity check: force full budget consumption by topping up n_init. Was missing before.
        if algo in ["shgo", "shgo_individual", "shgo_c", "shgo_c_3"]:
            gw = stored_params.get("grad_weight", 2)
            cost = _cost_shgo(
                stored_params["shgo_initial_samples"],
                stored_params["shgo_local_runs"],
                stored_params["shgo_local_steps"],
                gw,
            )
            if cost != budget:
                delta = budget - cost
                stored_params["shgo_initial_samples"] = max(
                    1,
                    stored_params["shgo_initial_samples"] + delta
                )
                new_cost = _cost_shgo(
                    stored_params["shgo_initial_samples"],
                    stored_params["shgo_local_runs"],
                    stored_params["shgo_local_steps"],
                    gw,
                )
                print(f"[{algo}] adjusted n_init by {delta} to match budget: {cost} -> {new_cost}")
                assert new_cost == budget, f"SHGO cost mismatch after fix: {new_cost}!={budget}"
                save_params(stored_params, param_path)

        # assert that grad weight matches
        if algo in grad_weight_algos:
            assert stored_params.get("grad_weight", None) == grad_weight, \
                f"Grad weight mismatch for {algo}: {stored_params.get('grad_weight', None)}!={grad_weight}"

        # Rebuild optimizer with best params
        search_obj = build_search_algorithm(
            algo_name_for_path,
            stored_params,
            problem=current_problems[0],  # any problem is fine for optimizer construction
            budget=budget,
            model=model,
        )
        print("Rebuilt search object from saved params.")

        for prob, method_name in zip(current_problems, current_problem_names):
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            print(f"[{method_name}] evaluating (cached path: {eval_path})")
            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=search_obj,
                problem=prob,
                test_loader=test_loader_transformed,
                save_path=eval_path,
                max_batch_override=dataset_info.batch_size_search,
                show_progress=show_progress,
                repeats=eval_repeats,
                return_per_run=True,
                overwrite=False, store_val=True, rerun_if_duplicates=True if not "its" in algo else False,
            )
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import json
import os

# Collect results
results = []

for budget in budgets:
    for algo in all_algos:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        for method_name in problem_names:
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            if os.path.exists(eval_path):
                with open(eval_path, "r") as f:
                    metrics = json.load(f)

                accuracy = metrics.get("accuracy_mean", None)
                accuracy_se = metrics.get("accuracy_se", None)
                accuracy_std = metrics.get("accuracy_std", None)
                number_of_runs = metrics.get("repeats", None)
                results.append({
                    "Algorithm": algo,
                    "Budget": budget,
                    "Problem": method_name,
                    "Accuracy": accuracy,
                    "Accuracy_SE": accuracy_se,
                    "Accuracy_STD": accuracy_std,
                    "Number_of_Runs": number_of_runs,

                })

# Convert to DataFrame
df = pd.DataFrame(results)





In [ ]:
#fancy renaming for plotting.

In [ ]:
ALGO_RENAME = {
    "cd": "CD-S",
    "cd_multi_cyclus": "CD",
    "shgo": "RS-LR",
    "parallel_sa": "PSA",
    "pso": "PSO",
    "pgd": "GD",
    "pgd_restart": "GD-R",
    "its": "ITS",
    "random_search": "R. Search",
    "wcd": "WCD",
    "wcd_lattice": "WCD-L",
    "shgo_c": "RS-LR-C",
    "pgd_c": "GD-C",
    "wcd_lat_ind": "WCD-L-EUL",
    "pgd_vector": "GD-Vec",
    "shgo_vector": "RS-LR-Vec",
}

PROBLEM_RENAME = {
    "knn_per_class": "PC-kNN",
    "logit_energy": "Logit Energy",
    "learned_energy": "Learned Energy",
}


In [ ]:
#do some renaming

df_renamed = df.copy()

df_renamed["Algorithm"] = df["Algorithm"].replace(ALGO_RENAME)

#reanme problem name from knn_per_class to PC-kNN
df_renamed["Problem"] = df["Problem"].replace(PROBLEM_RENAME)
problem_names_renamed = ["Logit Energy", "PC-kNN", "Learned Energy"]


In [ ]:
fullname_dict = {
    "RS-LR-Vec": "Random Search with Local Refinement (rotation vector representation)",
    "CD": "Coordinate Descent",
    "RS-LR": "Random Search with Local Refinement",
    "ITS": "Inverse Transformation Search",
    "R. Search": "Random Search",
    "PSA": "Parallel Simulated Annealing",
    "PSO": "Particle Swarm Optimization",
    "GD": "Multistart Gradient Descent",
    "WCD-L-EUL": "Weighted Coordinate Descent with Lattice sampling (euler angles)",
    "WCD-L": "Weighted Coordinate Descent with Lattice sampling",
    "WCD": "Weighted Coordinate Descent",
    "GD-R": "Multistart Gradient Descent with Restarts",
    "RS-LR-C": "Random Search with Local Refinement (complex/quaternion rotations)",
    "GD-C": "Multistart Gradient Descent (complex/quaternion rotations)",
    "GD-Vec": "Multistart Gradient Descent (rotation vector representation)",
    "CD-S": "Coordinate Descent (single cycle)",
}

In [ ]:
import matplotlib.pyplot as plt

short_names = list(fullname_dict.keys())
n = len(short_names)

# Get all 20 colors from tab20
tab20_colors = [plt.get_cmap("tab20")(i)[:3] for i in range(20)]

# Separate even and odd indices
even_colors = tab20_colors[::2]  # 0, 2, 4, ..., 18
odd_colors = tab20_colors[1::2]  # 1, 3, 5, ..., 19

# Combine: even first, then odd
ordered_colors = even_colors + odd_colors

# Repeat pattern if more names than 20
if n > 20:
    repeats = (n // 20) + 1
    ordered_colors = (ordered_colors * repeats)[:n]
else:
    ordered_colors = ordered_colors[:n]

# Assign to names
algorithm_colors = {name: ordered_colors[i] for i, name in enumerate(short_names)}



In [ ]:
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_search_datasets", dataset,
                           transform_name)
if not os.path.exists(figure_path):
    os.makedirs(figure_path)

In [ ]:
from matplotlib.patches import Patch
from src.utils.eval.vis import plt_setup_latex

W = plt_setup_latex()

# Sort algorithm keys alphabetically
sorted_shorts = sorted(fullname_dict.keys())

handles = [
    Patch(color=algorithm_colors[short], label=f"{short} — {fullname_dict[short]}")
    for short in sorted_shorts
]

# Plot legend in a separate figure
fig, ax = plt.subplots(figsize=(W, W * 0.4))
ax.legend(
    handles=handles,
    loc='center',
    frameon=False,
    fontsize=9
)
ax.axis('off')
plt.savefig(os.path.join(figure_path, "comparision_search_algorithms_legend.pdf"), bbox_inches='tight')
plt.savefig(os.path.join(figure_path, "comparision_search_algorithms_legend.pgf"), bbox_inches='tight')
plt.tight_layout()
plt.show()


In [ ]:
from src.utils.eval.vis import plt_setup_latex

W = plt_setup_latex()

In [ ]:
import seaborn as sns
import os


def plot_accuracy_bars(
        df,
        problem_column="Problem",
        algo_column="Algorithm",
        mean_column="Accuracy",
        se_column="Accuracy_SE",
        problem_names=None,
        algorithm_colors=None,
        figure_path=None,
        fig_width=None,
        fig_height=None,
        error_capsize=1.5,
        save_name="comparison_search_algorithms"
):
    """
    Plots horizontal bar charts of accuracy with error bars for each problem.
    Works for both individual problem scores and grouped scores.

    Parameters:
        df : pd.DataFrame
            Data containing algorithm performance.
        problem_column : str
            Column name for problems.
        algo_column : str
            Column name for algorithms.
        mean_column : str
            Column name for mean accuracy.
        se_column : str
            Column name for standard error.
        problem_names : list
            Ordered list of problems to plot. If None, all unique problems in df are used.
        algorithm_colors : dict
            Mapping from algorithm name to color (RGB tuple or string). Defaults to grey if missing.
        figure_path : str
            Directory to save the figure. If None, figure is not saved.
        fig_width, fig_height : float
            Figure dimensions. If None, defaults are computed automatically.
        error_capsize : float
            Capsize for error bars.
        save_name : str
            Base name for saved figure files (PDF & PGF).
    """

    if problem_names is None:
        problem_names = df[problem_column].unique()

    if algorithm_colors is None:
        algorithm_colors = {}

    n_problems = len(problem_names)

    if fig_width is None:
        fig_width = 8
    if fig_height is None:
        fig_height = fig_width / max(2.2, n_problems / 3)

    fig, axes = plt.subplots(
        1, n_problems,
        figsize=(fig_width, fig_height),
        sharey=False
    )

    if n_problems == 1:
        axes = [axes]

    for idx, (ax, problem) in enumerate(zip(axes, problem_names)):
        sub = df[df[problem_column] == problem].sort_values(algo_column)

        algos = sub[algo_column].tolist()
        accs = sub[mean_column].to_numpy()
        ses = sub[se_column].to_numpy() * 1.96
        y = np.arange(len(algos))

        colors = [algorithm_colors.get(a, (0.5, 0.5, 0.5)) for a in algos]

        ax.barh(
            y, accs, xerr=ses,
            capsize=error_capsize,
            color=colors,
            edgecolor="none",
            height=0.7,
            alpha=1,
            zorder=2,
            error_kw={'elinewidth': 1.0, 'ecolor': 'black', "capthick": 1.0}

        )

        ax.set_yticks(y)
        if idx == 0:
            ax.set_yticklabels(algos, fontsize=8)
            if dataset == "modelnet10":
                ax.set_yticklabels(algos, fontsize=8)

            ax.set_ylabel("Algorithm", fontsize=10)
        else:
            ax.set_yticklabels([])

        ax.invert_yaxis()  # first algorithm on top

        xmin = max(0, (accs - ses).min() - 0.03)
        xmax = min(1, (accs + ses).max() + 0.03)
        ax.set_xlim(xmin, xmax)

        ax.set_xlabel("Accuracy", fontsize=10)
        ax.set_title(problem, fontsize=11)
        ax.grid(True, axis='x', linestyle='--', alpha=0.7, zorder=0)
        sns.despine(ax=ax, left=True, bottom=False)
        #disabel y tick markers -
        ax.tick_params(axis='y', length=0)

    #plt.tight_layout(pad=0.5, w_pad=0.2)

    if figure_path is not None:
        os.makedirs(figure_path, exist_ok=True)
        plt.savefig(os.path.join(figure_path, f"{save_name}.pdf"), bbox_inches='tight')
        plt.savefig(os.path.join(figure_path, f"{save_name}.pgf"), bbox_inches='tight')
    plt.tight_layout()
    plt.show()


plot_accuracy_bars(
    df=df_renamed,
    problem_names=problem_names_renamed,
    algorithm_colors=algorithm_colors,
    figure_path=figure_path,
    fig_width=W,
)


In [ ]:



def compute_grouped_scores(group):
    """
    Compute combined mean and SE across problems.
    """
    N = len(group)  # number of problems
    # Mean across problems
    mean_accuracy = group["Accuracy"].mean()

    # Variance of each problem mean: s_i^2 / n_i
    var_each = (group["Accuracy_STD"].to_numpy() ** 2) / group["Number_of_Runs"].to_numpy()

    # Combined SE across problems
    se_combined = np.sqrt(np.sum(var_each)) / N

    return pd.Series({
        "Grouped_Accuracy_Mean": mean_accuracy,
        "Grouped_Accuracy_SE": se_combined
    })


# Apply grouping
grouped_scores = df_renamed.groupby(["Algorithm", "Budget"]).apply(compute_grouped_scores).reset_index()
grouped_scores["Problem"] = "Grouped"
print(grouped_scores)

#plot grouped scores

In [ ]:
plot_accuracy_bars(
    df=grouped_scores,  # DataFrame with Grouped_Accuracy_Mean & Grouped_Accuracy_SE
    mean_column="Grouped_Accuracy_Mean",
    se_column="Grouped_Accuracy_SE",
    problem_names=None,  # will use all problems if None
    algorithm_colors=algorithm_colors,
    figure_path=figure_path,
    save_name="grouped_comparison_search_algorithms",
    fig_width=W / 2,
    fig_height=W / 2,
)


In [ ]:
#plot grouped accuracy across problems


In [ ]:
results_list = []
for budget in ["60"]:
    for algo in all_algos:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        for method_name in problem_names:
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            if os.path.exists(eval_path):
                with open(eval_path, "r") as f:
                    metrics = json.load(f)
                results_list.append({
                    "Algorithm": algo,
                    "Problem": method_name,
                    "Budget": budget,
                    "metrics": metrics
                })

In [ ]:
#Code that reads the results of all runs(including resulst for each sample in the dataset) and calculates relative error metric.

In [ ]:
def analyze_run_results_with_budget(results_list):
    """
    Computes summary statistics per algorithm and budget for each problem.
    Uses global min-max normalization across all budgets, runs, and algorithms.
    Frobenius distance is computed against the global best run per sample.
    """
    import numpy as np
    from collections import defaultdict

    # Apply renaming first, then group
    for entry in results_list:
        algo = ALGO_RENAME.get(entry["Algorithm"], entry["Algorithm"])
        problem_name = PROBLEM_RENAME.get(entry["Problem"], entry["Problem"])
        entry["Algorithm"] = algo
        entry["Problem"] = problem_name

    results_by_problem = defaultdict(list)
    for entry in results_list:
        results_by_problem[entry["Problem"]].append(entry)

    problem_summaries = {}

    for problem, entries in results_by_problem.items():
        # First pass: collect all errors globally to find min/max and best run per sample
        all_errors_global = []
        all_mats_global = []
        metadata_global = []
        all_true_labels, all_pred_labels = [], []

        for entry in entries:
            algo = entry["Algorithm"]
            problem_name = entry["Problem"]
            budget = entry["Budget"]

            res = entry["metrics"]
            runs = res.get("per_run", [res])

            for run_idx, run in enumerate(runs):
                if "per_sample_errors" not in run or "per_sample_matrices" not in run:
                    continue

                errs = np.array(run["per_sample_errors"], dtype=float)
                mats = run["per_sample_matrices"]

                all_errors_global.append(errs)
                all_mats_global.append(mats)
                metadata_global.append({
                    'algo': algo,
                    'budget': budget,
                    'run_idx': run_idx
                })

                if "per_sample_true_labels" in run:
                    all_true_labels.append(run["per_sample_true_labels"])
                if "per_sample_pred_labels" in run:
                    all_pred_labels.append(run["per_sample_pred_labels"])

        if not all_errors_global:
            continue

        array_labels = np.array(all_true_labels)
        assert np.all(array_labels == array_labels[0]), "Rows are not equal"

        all_errors_global = np.stack(all_errors_global)
        eps = 1e-12
        num_samples = all_errors_global.shape[1]

        # Global min/max across all budgets and algorithms
        min_errors = np.nanmin(all_errors_global, axis=0)
        max_errors = np.nanmax(all_errors_global, axis=0)
        denom = max_errors - min_errors + eps

        # Global best run per sample
        best_run_idx_per_sample = np.nanargmin(
            np.where(np.isnan(all_errors_global), np.inf, all_errors_global), axis=0
        )
        best_mats = [all_mats_global[idx][j] for j, idx in enumerate(best_run_idx_per_sample)]

        # Best predicted labels
        best_pred_labels = None
        pred_labels_available = len(all_pred_labels) == len(all_errors_global)
        if pred_labels_available:
            best_pred_labels = [
                all_pred_labels[idx][j] for j, idx in enumerate(best_run_idx_per_sample)
            ]

        # Vectorized relative error computation
        relative_errors_global = (all_errors_global - min_errors[None, :]) / denom[None, :]

        # Pre-compute Frobenius distances

        best_mats_array = np.array(best_mats, dtype=float)
        all_mats_stacked = []
        for run_mats in all_mats_global:
            run_mats_array = np.array(run_mats[:num_samples], dtype=float)
            all_mats_stacked.append(run_mats_array)
        all_mats_stacked = np.array(all_mats_stacked)

        diff = all_mats_stacked - best_mats_array[None, :, ...]
        shape = diff.shape
        diff_flat = diff.reshape(shape[0], shape[1], -1)
        best_flat = best_mats_array.reshape(shape[1], -1)

        diff_norms = np.linalg.norm(diff_flat, axis=2)
        best_norms = np.linalg.norm(best_flat, axis=1)

        frobenius_distances_global = diff_norms / (best_norms[None, :] + eps)
        valid_mask = ~np.isnan(all_errors_global) & np.isfinite(frobenius_distances_global)
        frobenius_distances_global = np.where(valid_mask, frobenius_distances_global, np.nan)

        # Second pass: organize by budget and algorithm
        summary_by_budget = defaultdict(dict)

        for entry in entries:
            algo = entry["Algorithm"]
            budget = entry["Budget"]
            res = entry["metrics"]
            runs = res.get("per_run", [res])

            # Find indices for this algo+budget combination
            indices = [i for i, meta in enumerate(metadata_global)
                       if meta['algo'] == algo and meta['budget'] == budget]

            if not indices:
                continue

            algo_rel_errs = relative_errors_global[indices]
            algo_frobs = frobenius_distances_global[indices]

            run_avg_rel_errs = np.nanmean(algo_rel_errs, axis=1)
            run_avg_rel_errs = run_avg_rel_errs[~np.isnan(run_avg_rel_errs)]

            run_avg_frobs = np.nanmean(algo_frobs, axis=1)
            run_avg_frobs = run_avg_frobs[~np.isnan(run_avg_frobs)]

            num_runs = len(run_avg_rel_errs)

            accuracy = res["accuracy_mean"] if "accuracy_mean" in res else None
            accuracy_se = res["accuracy_se"] if "accuracy_se" in res else None

            summary_by_budget[budget][algo] = {
                "accuracy": accuracy,
                "accuracy_se": accuracy_se,
                "mean_relative_error": float(np.mean(run_avg_rel_errs)) if len(run_avg_rel_errs) > 0 else None,
                "std_relative_error": float(np.std(run_avg_rel_errs, ddof=1)) if num_runs > 1 else 0,
                "mean_frobenius": float(np.mean(run_avg_frobs)) if len(run_avg_frobs) > 0 else None,
                "std_frobenius": float(np.std(run_avg_frobs, ddof=1)) if num_runs > 1 else 0,
                "se_relative_error": float(np.std(run_avg_rel_errs, ddof=1) / np.sqrt(num_runs)) if num_runs > 1 else 0,
                "se_frobenius": float(np.std(run_avg_frobs, ddof=1) / np.sqrt(num_runs)) if len(
                    run_avg_frobs) > 1 else 0,
                "num_runs": num_runs
            }

        problem_summaries[problem] = {
            "num_datapoints": num_samples,
            "budgets": dict(summary_by_budget),
            "true_labels": all_true_labels[0],
        }

    return problem_summaries


def analyze_run_results(results_list):
    """
    Wrapper around analyze_run_results_with_budget that assumes a budget for all results.
    """
    budget_added = False
    if results_list and "Budget" not in results_list[0]:
        for entry in results_list:
            if "Budget" not in entry:
                entry["Budget"] = 0  # dummy budget
        budget_added = True

    # This computes the summary for the single budget group.
    full_summary = analyze_run_results_with_budget(results_list)

    collapsed_summary = {}

    for problem, summary in full_summary.items():
        budgets = summary["budgets"]

        if not budgets:
            algo_summary = {}
        else:
            budget_key = next(iter(budgets))
            algo_summary = budgets[budget_key]

        collapsed_summary[problem] = {
            "num_datapoints": summary["num_datapoints"],
            "true_labels": summary.get("true_labels"),
            "algorithms": algo_summary,
        }

    if budget_added:
        for entry in results_list:
            if "Budget" in entry and entry["Budget"] == 0:
                del entry["Budget"]

    return collapsed_summary





In [ ]:
analysis = analyze_run_results(results_list)

In [ ]:
# Print summary per problem
for problem, pdata in analysis.items():
    print(f"\n=== Problem: {problem} ===")
    #print(f"Analyzed {pdata['num_datapoints']} datapoints across {pdata['num_runs']} runs")
    for algo, stats in pdata["algorithms"].items():
        print(f"  Algorithm: {algo}")
        print(f"    Mean relative error:   {stats['mean_relative_error']:.6f} ± {stats['std_relative_error']:.6f}")
        print(f"    Mean Frobenius dist.:  {stats['mean_frobenius']:.6f} ± {stats['std_frobenius']:.6f}")
        print(f"    Accuracy:              {stats.get('accuracy', 'N/A')} ± {stats.get('accuracy_se', 'N/A')}")

In [ ]:
import numpy as np

for entry in results_list:
    algo = entry["Algorithm"]
    prob = entry["Problem"]
    metrics = entry["metrics"]

    runs = metrics.get("per_run", [metrics])
    for run_idx, run in enumerate(runs):
        mats = run.get("per_sample_matrices", [])
        for j, m in enumerate(mats):
            m_arr = np.array(m, dtype=float)
            if np.any(np.isnan(m_arr)):
                print(f"NaN matrix found: Problem={prob}, Algo={algo}, Run={run_idx}, Sample={j}")
                break

In [ ]:



def analysis_to_df(analysis):
    """
    Convert the nested 'analysis' dict from analyze_run_results
    into a long DataFrame suitable for plotting.
    """
    rows = []
    for problem, pdata in analysis.items():
        for algo, stats in pdata["algorithms"].items():
            rows.append({
                "Problem": problem,
                "Algorithm": algo,
                "mean_relative_error": stats.get("mean_relative_error"),
                "se_relative_error": stats.get("se_relative_error"),
                "std_relative_error": stats.get("std_relative_error"),
                "mean_frobenius": stats.get("mean_frobenius"),
                "se_frobenius": stats.get("se_frobenius"),
                "std_frobenius": stats.get("std_frobenius"),
                "num_runs": stats.get("num_runs"),
                "num_datapoints": pdata.get("num_datapoints")
            })
    df = pd.DataFrame(rows)
    return df


In [ ]:
df_analysis = analysis_to_df(analysis)
df_analysis

In [ ]:



def plot_df_analysis(df, metric="mean_relative_error", error_col=None):
    """
    Plot results from a flattened analysis DataFrame.

    df: pd.DataFrame with columns ['Problem', 'Algorithm', metric, error_col]
    metric: which mean to plot ('mean_relative_error' or 'mean_frobenius')
    error_col: which column contains the error bars (SEM)
    """
    if error_col is None:
        error_col = "se_relative_error" if metric == "mean_relative_error" else "se_frobenius"

    problem_names = df["Problem"].unique()
    num_problems = len(problem_names)

    fig, axes = plt.subplots(1, num_problems, figsize=(6 * num_problems, 5), squeeze=False)

    for ax, problem in zip(axes[0], problem_names):
        sub = df[df["Problem"] == problem].sort_values("Algorithm")
        algos = sub["Algorithm"].tolist()
        means = sub[metric].tolist()
        errors = sub[error_col].tolist()

        x = np.arange(len(algos))
        ax.bar(x, means, yerr=errors, capsize=5, alpha=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(algos, rotation=45, ha="right")
        ax.set_ylabel(metric.replace("_", " ").title())
        ax.set_title(f"Problem: {problem}")
        ax.grid(axis="y", linestyle="--", alpha=0.7)

    #plt.tight_layout()
    plt.show()


plot_df_analysis(df_analysis, metric="mean_relative_error")

In [ ]:
plot_df_analysis(df_analysis, metric="mean_frobenius")

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os


def plot_analysis_with_sem(analysis, algorithm_colors, metric="mean_relative_error", W=8, savepath=None):
    """
    Plot per-problem results with horizontal bars and SEM error bars.
    Total figure width is W, height is W/2.2.

    Parameters:
    -----------
    analysis : dict or pd.DataFrame
        Dictionary from analyze_run_results OR a flattened DataFrame with columns:
        ['Problem', 'Algorithm', metric, se_relative_error, se_frobenius']
    algorithm_colors : dict
        Mapping from algorithm name to RGB color tuple
    metric : str
        Which mean to plot ("mean_relative_error" or "mean_frobenius")
    W : float
        Total figure width in inches
    savepath : str
        Path to save the figure (optional)
    """
    # Determine if input is dict or DataFrame
    if isinstance(analysis, dict):
        problem_names = list(analysis.keys())
    elif isinstance(analysis, pd.DataFrame):
        problem_names = analysis["Problem"].unique()
    else:
        raise ValueError("analysis must be a dict or a pandas DataFrame")

    num_problems = len(problem_names)
    if isinstance(W, (int, float)):
        fig_width = W
        fig_height = W / 2.2  # proportional height
    else:
        fig_width, fig_height = W

    fig, axes = plt.subplots(
        1, num_problems,
        figsize=(fig_width, fig_height),
        sharey=False,
        squeeze=False
    )
    axes = axes[0]  # flatten if necessary

    # Metric label
    metric_label = metric.replace("_", " ").title()
    if metric == "mean_relative_error":
        metric_label = "Relative Error"
    elif metric == "mean_frobenius":
        metric_label = "Frobenius Error"

    for idx, ax in enumerate(axes):
        problem = problem_names[idx]

        if isinstance(analysis, dict):
            pdata = analysis[problem]
            algos = sorted(pdata["algorithms"].keys())
            means = []
            sems = []
            for algo in algos:
                stats = pdata["algorithms"][algo]
                means.append(stats[metric])
                sem_val = (
                    stats["se_relative_error"] if metric == "mean_relative_error"
                    else stats["se_frobenius"]
                )
                sems.append(sem_val)
        else:  # DataFrame
            sub = analysis[analysis["Problem"] == problem].sort_values("Algorithm")
            algos = sub["Algorithm"].tolist()
            means = sub[metric].tolist()
            sems = sub["se_relative_error"].tolist() if metric == "mean_relative_error" else sub[
                "se_frobenius"].tolist()

        y = np.arange(len(algos))
        colors = [algorithm_colors.get(a, (0.5, 0.5, 0.5)) for a in algos]
        sems = [sem * 1.96 for sem in sems]
        ax.barh(
            y, means, xerr=sems,
            capsize=1.5,
            color=colors,
            edgecolor="none",
            height=0.7,
            alpha=1,
            error_kw={'elinewidth': 1.0, 'ecolor': 'black', "capthick": 1.0}
        )

        ax.set_yticks(y)
        if idx == 0:
            ax.set_yticklabels(algos, fontsize=8)
            if dataset == "modelnet10":
                ax.set_yticklabels(algos, fontsize=8)

            ax.set_ylabel("Algorithm", fontsize=10)
        else:
            ax.set_yticklabels([])

        ax.invert_yaxis()
        means_arr = np.array(means)
        sems_arr = np.array([s if s is not None else 0 for s in sems])
        xmin = max(0, (means_arr - sems_arr).min() - 0.03 * means_arr.max())
        xmax = (means_arr + sems_arr).max() + 0.03 * means_arr.max()
        ax.set_xlim(xmin, xmax)

        ax.set_xlabel(metric_label, fontsize=10)
        ax.set_title(f"{problem}", fontsize=11)
        ax.grid(True, axis='x', linestyle='--', alpha=0.7)
        sns.despine(ax=ax, left=True, bottom=False)
        ax.tick_params(axis='y', length=0)

    #plt.tight_layout(pad=0.5, w_pad=0.2)
    if savepath is not None:
        os.makedirs(os.path.dirname(savepath), exist_ok=True)
        plt.savefig(savepath, bbox_inches='tight')
    plt.tight_layout()
    plt.show()


path_pdf = os.path.join(figure_path, "mean_relative_error.pdf")
plot_analysis_with_sem(analysis, algorithm_colors, metric="mean_relative_error", W=W, savepath=path_pdf)

path_pgf = os.path.join(figure_path, "mean_relative_error.pgf")
plot_analysis_with_sem(analysis, algorithm_colors, metric="mean_relative_error", W=W, savepath=path_pgf)


In [ ]:
path = os.path.join(figure_path, f"mean_frobenius.pdf")
plot_analysis_with_sem(analysis, algorithm_colors, metric="mean_frobenius", savepath=path, W=W)
path = os.path.join(figure_path, f"mean_frobenius.pgf")
plot_analysis_with_sem(analysis, algorithm_colors, metric="mean_frobenius", savepath=path, W=W)

In [ ]:
def compute_grouped_scores(df):
    grouped = df.groupby("Algorithm").agg(
        mean_relative_error=("mean_relative_error", "mean"),
        se_relative_error=("se_relative_error", lambda x: np.sqrt(np.sum(x ** 2)) / len(x)),
        mean_frobenius=("mean_frobenius", "mean"),
        se_frobenius=("se_frobenius", lambda x: np.sqrt(np.sum(x ** 2)) / len(x))
    ).reset_index()

    grouped["Problem"] = "Grouped"
    return grouped


df_analysis_grouped = compute_grouped_scores(df_analysis)

In [ ]:
df_analysis_grouped

In [ ]:
#now plot grouped scores with sem
plot_analysis_with_sem(
    df_analysis_grouped,
    algorithm_colors,
    metric="mean_relative_error",
    savepath=os.path.join(figure_path, "grouped_mean_relative_error.pdf"),
    W=(W / 2, W / 2)
)
plot_analysis_with_sem(
    df_analysis_grouped,
    algorithm_colors,
    metric="mean_relative_error",
    savepath=os.path.join(figure_path, "grouped_mean_relative_error.pgf"),
    W=(W / 2, W / 2)
)

In [ ]:
plot_analysis_with_sem(
    df_analysis_grouped,
    algorithm_colors,
    metric="mean_frobenius",
    savepath=os.path.join(figure_path, "grouped_mean_frobenius.pdf"),
    W=(W / 2, W / 2.2),
)
plot_analysis_with_sem(
    df_analysis_grouped,
    algorithm_colors,
    metric="mean_frobenius",
    savepath=os.path.join(figure_path, "grouped_mean_frobenius.pgf"),
    W=(W / 2, W / 2.2),
)

In [ ]:
for budget in budgets:
    print(f"\n=== Best Params Summary | Budget: {budget} ===")
    for algo in all_algos:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        param_path = os.path.join(algo_dir, "best.yml")

        if os.path.exists(param_path):
            stored_params = load_params(param_path)
            print(f"\n--- {algo} ---")
            for k, v in stored_params.items():
                print(f"{k}: {v}")
        else:
            print(f"\n--- {algo} ---")
            print("No stored params found.")


Here we recheck wether the eval count holds true for the algorithms. As well as the speed from forward vs forward plus backward.

In [ ]:
#disable p grad
for param in model.parameters():
    param.requires_grad = False

In [ ]:
class EvalCounterWrapper(nn.Module):
    def __init__(self, model):
        super(EvalCounterWrapper, self).__init__()
        self.model = model
        self.eval_count = 0

    def __call__(self, *args, **kwargs):
        batch = args[0] if args else kwargs.get("x", None)
        batch_size = len(batch) if batch is not None else 1

        # Count once if in no_grad, twice otherwise
        if torch.is_grad_enabled():
            self.eval_count += 2 * batch_size
        else:
            self.eval_count += batch_size

        return self.model(*args, **kwargs)

    def reset(self):
        self.eval_count = 0

In [ ]:

import os
import json
import matplotlib.pyplot as plt
import numpy as np
import statistics

# --- Setup ---
for param in model.parameters():
    param.requires_grad = False

# Assuming test_loader_transformed and device are predefined
x = next(iter(test_loader_transformed))[0].to(device)


def timing_cache_path(modelname, transform_name, dataset, architecture):
    base = os.path.join(current_path, "experiment_files", "export", "results", "timing_forward_backward_new")
    os.makedirs(base, exist_ok=True)
    fname = f"{dataset}_{architecture}_{modelname}_{transform_name}.json"
    return os.path.join(base, fname)


cache_path = timing_cache_path(modelname, transform_name, dataset, architecture)


def run_timing(model, x, n_trials=10, n_warmup=100, n_iter=1000):
    model.eval()
    if dataset == "tu_berlin":
        model.train()
    for param in model.parameters():
        param.requires_grad = False

    # Initialize CUDA events for high-precision GPU timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    forward_trials = []
    fb_trials = []

    for trial in range(n_trials):
        # --- 1. Measure Forward ---
        with torch.no_grad():
            # Warmup
            for _ in range(n_warmup):
                _ = model(x)
                torch.cuda.synchronize()
            torch.cuda.synchronize()

            # Timing Loop
            start_event.record()
            for _ in range(n_iter):
                _ = model(x)
                torch.cuda.synchronize()
            torch.cuda.synchronize()
            end_event.record()
            torch.cuda.synchronize()
            forward_trials.append((start_event.elapsed_time(end_event) / 1000) / n_iter)

        # --- 2. Measure Forward + Backward ---
        # Warmup
        for _ in range(n_warmup):
            x_tmp = x.detach().clone().requires_grad_(True)
            model(x_tmp).sum().backward()
            torch.cuda.synchronize()
        torch.cuda.synchronize()

        # Timing Loop
        start_event.record()
        for _ in range(n_iter):
            x_tmp = x.detach().clone().requires_grad_(True)
            model(x_tmp).sum().backward()
            torch.cuda.synchronize()
        torch.cuda.synchronize()
        end_event.record()

        torch.cuda.synchronize()
        fb_trials.append((start_event.elapsed_time(end_event) / 1000) / n_iter)

        print(f"Trial {trial + 1}/{n_trials} complete.")

    avg_f = statistics.median(forward_trials)
    avg_fb = statistics.median(fb_trials)

    print(f"Forward times: {forward_trials}")
    print(f"F+B times: {fb_trials}")
    if dataset == "tu_berlin":
        model.eval()
    return avg_f, avg_fb


# --- Run or load cache ---
if os.path.exists(cache_path) and True:  # Toggle True to use cache
    with open(cache_path, "r") as f:
        timing = json.load(f)
    res1, res2 = timing["forward"], timing["forward_backward"]
else:
    # Optional logic for specific datasets
    if dataset == "tu_berlin":
        model.train()

    res1, res2 = run_timing(model, x)

    if dataset == "tu_berlin":
        model.eval()

    with open(cache_path, "w") as f:
        json.dump({"forward": res1, "forward_backward": res2}, f)

# --- Plot ---
W = plt_setup_latex()
labels = ["Forward", "Forward+Backward"]

# Converting results to ms for plotting
means = [res1 * 1000, res2 * 1000]

# Compute backward overhead
factor = means[1] / means[0]

plt.figure(figsize=(W / 2, W / 3))
bars = plt.bar(labels, means, color=["#4C72B0", "#DD8452"], alpha=0.85)
plt.ylabel("Time (ms)")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Write overhead above the second bar
plt.text(
    x=1,
    y=means[1],
    s=f"{factor:.2f}×",
    ha='center',
    va='bottom',
    fontweight='bold'
)

# Save as PGF + PDF
save_path = os.path.join(figure_path, "timing_forward_backward_plot_new")
plt.savefig(save_path + ".pgf", bbox_inches="tight")
plt.savefig(save_path + ".pdf", bbox_inches="tight")
plt.tight_layout()

plt.show()

In [ ]:
#new result path with new gpu used for paper.
import time
import os
import torch
import json

from search.objective_generators import _copy_problem_with_init_method

num_batches = 50
num_warmup = 10  # Explicit warmup batch counter
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_warmup = next(iter(test_loader_transformed))[0].to(device)
with torch.no_grad():
    _ = model(batch_warmup)
print("Warm-up done.\n")

benchmark_results = {}

for budget in budgets:
    print(f"\n=== Budget: {budget} ===")
    for algo in all_algos:
        print(f"\n--- Algorithm: {algo} ---")

        # Select problems and names based on algorithm type
        if algo in complex_algos:
            current_problems = problems_complex
            current_problem_names = problem_names_complex
        elif algo in ["shgo_individual", "wcd_lat_ind"]:
            current_problems = problems_individual
            current_problem_names = problem_names_individual
        elif "vector" in algo:
            current_problems = problems_rotvec
            current_problem_names = problem_names_rotvec
        else:
            current_problems = problems
            current_problem_names = problem_names

        # Map algorithm names for construction
        if algo in complex_algos:
            algo_name_for_path = algo_variant_mapping[algo]
        elif algo == "shgo_individual":
            algo_name_for_path = "shgo"
        elif algo == "wcd_lat_ind":
            algo_name_for_path = "wcd_lattice"
        elif "vector" in algo:
            algo_name_for_path = algo.replace("_vector", "")
        else:
            algo_name_for_path = algo

        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        os.makedirs(algo_dir, exist_ok=True)
        param_path = os.path.join(algo_dir, "best.yml")
        cache_path = os.path.join(algo_dir, "benchmark_cache_new.json")
        stored_params = load_params(param_path) if os.path.exists(param_path) else get_default_params(
            algo_name_for_path, budget)

        # Load cache if exists
        if os.path.exists(cache_path):
            with open(cache_path, "r") as f:
                cache = json.load(f)
        else:
            cache = {}

        search_obj = build_search_algorithm(
            algo_name_for_path,
            stored_params,
            problem=current_problems[0],
            budget=budget,
            model=model,
        )

        for prob, prob_name in zip(current_problems, current_problem_names):
            prob.max_batch_size = dataset_info.batch_size_search

            if algo == "wcd_lat_ind":
                print("Setting lattice basis for WCD Lattice Individual")
                prob = _copy_problem_with_init_method(prob, init_method="permuted_lattice")
            if algo in ["its", "its2", "shgo_individual", "wcd_lat_ind"]:
                prob = ITSWRAPPER._prepare_problem(prob)

            if "vector" in algo:
                prop = replace_rotation_transforms_2vec(prob)

            cache_key = f"{prob_name}"
            if cache_key in cache:
                avg_time = cache[cache_key]["avg_time_per_batch"]
                print(f"[{prob_name}] Avg time (cached): {avg_time:.4f}s")
            else:
                data_iter = iter(test_loader_transformed)

                # 1. Warmup Phase
                print(f"[{prob_name}] Warming up for {num_warmup} batches...")
                for i in range(num_warmup):
                    try:
                        x_warmup = next(data_iter)[0].to(device)
                        search_obj.optimize(prob, x_warmup)
                    except StopIteration:
                        data_iter = iter(test_loader_transformed)

                # 2. Measurement Phase
                print(f"[{prob_name}] Measuring {num_batches} batches...")
                torch.cuda.synchronize()

                # Since search algorithms have CPU logic, used perf counter. Not sure if this is the best choice.
                t_start = time.perf_counter()

                for _ in range(num_batches):
                    try:
                        x_batch = next(data_iter)[0].to(device)
                    except StopIteration:
                        data_iter = iter(test_loader_transformed)
                        x_batch = next(data_iter)[0].to(device)

                    search_obj.optimize(prob, x_batch)

                if device.type == 'cuda':
                    torch.cuda.synchronize()

                t_end = time.perf_counter()

                avg_time = (t_end - t_start) / num_batches

                # Store and Save
                cache[cache_key] = {"avg_time_per_batch": avg_time}
                with open(cache_path, "w") as f:
                    json.dump(cache, f)
                print(f"[{prob_name}] Avg time: {avg_time:.4f}s")

            benchmark_results.setdefault(algo, {})[prob_name] = {
                "avg_time_per_batch": avg_time,
            }
        torch.cuda.empty_cache()


In [ ]:

import os
import torch
import json
import pandas as pd

#speedtest
datasets = ["bigger_mnist", "bigger_emnist", "modelnet10", "tu_berlin"]
num_batches = 50
num_warmup = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_algos_across_datasets = ["shgo", "parallel_sa",
                             "pso", "cd", "wcd",
                             "pgd", "its", "random_search", "cd_multi_cyclus", "wcd_lattice"
                             ]

all_algos_across_datasets.append("wcd_lat_ind")

# Store results for all datasets
all_results = {}

for ds in datasets:
    print(f"\n{'=' * 80}")
    print(f"Processing Dataset: {ds}")
    print(f"{'=' * 80}\n")

    architecture = default_architecutre_mapping[ds]
    dataset_info = get_dataset_info(ds)
    transform_name = dataset_info.transform_seq_name

    base_results_dir = os.path.join(
        current_path, "experiment_files", "search_results",
        str(ds), transform_name, architecture
    )
    os.makedirs(base_results_dir, exist_ok=True)

    benchmark_results = {}
    problem_names = ["logit_energy", "knn_per_class", "learned_energy"]

    for budget in budgets:
        print(f"\n=== Budget: {budget} ===")
        for algo in all_algos_across_datasets:
            print(f"--- Algorithm: {algo} ---")

            algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
            os.makedirs(algo_dir, exist_ok=True)

            cache_path = os.path.join(algo_dir, "benchmark_cache_new.json")

            if os.path.exists(cache_path):
                with open(cache_path, "r") as f:
                    algo_results = json.load(f)
                if "its" in algo:
                    for prob in problem_names:
                        if prob == "learned_energy":
                            time_per_batch = algo_results[prob].get("avg_time_per_batch", 0)
                            algo_results[prob]["avg_time_per_batch"] = time_per_batch / 2
                print(f"Loaded cached results from {cache_path}")
            else:
                algo_results = {
                    prob: {"avg_time_per_batch": 0.0}
                    for prob in problem_names
                }
                print(f"No cache found at {cache_path} - using placeholder values")

            benchmark_results[algo] = algo_results

    all_results[ds] = benchmark_results

# Merge wcd_lattice and wcd_lat_ind: keep the values where the other is missing.
for ds in datasets:
    results = all_results[ds]

    lattice = results.get("wcd_lattice", {})
    lat_ind = results.get("wcd_lat_ind", {})

    merged = {}
    for prob in ["logit_energy", "knn_per_class", "learned_energy"]:
        merged[prob] = {}
        # Take from lattice if exists, else from lat_ind
        if lattice.get(prob, {}).get("avg_time_per_batch", 0):
            merged[prob] = lattice[prob]
        elif lat_ind.get(prob, {}).get("avg_time_per_batch", 0):
            merged[prob] = lat_ind[prob]
        else:
            merged[prob] = {"avg_time_per_batch": np.nan}

    results["wcd_lattice"] = merged
    # Remove the separate lat_ind entry
    if "wcd_lat_ind" in results:
        del results["wcd_lat_ind"]

    # Remove shgo results for tu_berlin
    if ds == "tu_berlin" and "shgo" in results:
        del results["shgo"]
    if ds == "tu_berlin" and "pgd" in results:
        del results["pgd"]

# Create summary table across all datasets
summary_rows = []
for dataset_name, results in all_results.items():
    for algo, problems_dict in results.items():
        if problems_dict:
            times = [v.get("avg_time_per_batch", 0) for v in problems_dict.values()]
            summary_rows.append({
                "Dataset": dataset_name,
                "Algorithm": algo,
                "Mean_Time_s": sum(times) / len(times) if times else 0,
            })

df_summary = pd.DataFrame(summary_rows).sort_values(["Dataset", "Mean_Time_s"])

df_summary["Algorithm"] = df_summary["Algorithm"].replace(ALGO_RENAME)

print("\n" + "=" * 80)
print("BENCHMARK SUMMARY - ALL DATASETS")
print("=" * 80)
print(df_summary.to_string(index=False))
#beaty names bigger_mnist -> MNIST, bigger_emnist -> EMNIST, modelnet10 -> ModelNet10, tu_berlin -> TU Berlin
df_summary["Dataset"] = df_summary["Dataset"].replace({
    "bigger_mnist": "MNIST",
    "bigger_emnist": "EMNIST",
    "modelnet10": "ModelNet10",
    "tu_berlin": "TU Berlin"
})
dataset_order = ["MNIST", "EMNIST", "ModelNet10", "TU Berlin"]
df_summary["Dataset"] = pd.Categorical(df_summary["Dataset"], categories=dataset_order, ordered=True)
df_summary = df_summary.sort_values(["Dataset", "Mean_Time_s"])
# Pivot the DataFrame: Algorithms as rows (index), Datasets as columns
df_pivot = df_summary.pivot(index="Algorithm", columns="Dataset", values="Mean_Time_s")



# Generate the LaTeX table string
latex_table = df_pivot.to_latex(
    float_format="%.4f",
    caption="Mean processing time per batch (seconds) across different datasets and algorithms.",
    label="tab:benchmark_results",
    column_format="l" + "c" * len(datasets),  # left-align algorithm names, center-align data
    position="ht"
)

# Print and save
print("\n" + "=" * 80)
print("LATEX TABLE")
print("=" * 80)
print(latex_table)

latex_path = os.path.join(current_path, "experiment_files", "export", "benchmark_summary.tex")
with open(latex_path, "w") as f:
    f.write(latex_table)

print(f"\nLaTeX table saved to: {latex_path}")


In [ ]:
baseline_algo = "R. Search"

if baseline_algo in df_pivot.index:
    # 1. Calculate Relative Time: Time_algo / Time_random
    # (Values > 1 means slower than random, < 1 means faster)
    df_relative = df_pivot.div(df_pivot.loc[baseline_algo], axis=1)

    # 2. Calculate Average Relative Time across all datasets (rows)
    df_relative['Average_Relative'] = df_relative.mean(axis=1)

    print("\n" + "=" * 80)
    print("RELATIVE TIME TO RANDOM SEARCH (1.0 = Baseline)")
    print("=" * 80)
    print(df_relative.to_string())

    # 3. Export to LaTeX
    relative_latex = df_relative.to_latex(
        float_format="%.3f",
        caption="Relative processing time compared to Random Search across datasets.",
        label="tab:relative_benchmark",
        position="ht"
    )

    rel_path = os.path.join(current_path, "experiment_files", "export", "relative_benchmark.tex")
    with open(rel_path, "w") as f:
        f.write(relative_latex)
else:
    print(f"Warning: Baseline '{baseline_algo}' not found in algorithms.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

rows = []
for algo, prob_dict in benchmark_results.items():
    for prob_name, metrics in prob_dict.items():
        rows.append({
            "Algorithm": algo,
            "Problem": prob_name,
            "AvgTimePerBatch": metrics["avg_time_per_batch"]
        })
df_time = pd.DataFrame(rows)

df_time["Algorithm"] = df_time["Algorithm"].replace(ALGO_RENAME)
df_time["Problem"] = df_time["Problem"].replace(PROBLEM_RENAME)

# Adjust for Learned-Energy cases
df_time.loc[
    (df_time["Problem"] == "Learned Energy") & (df_time["Algorithm"].str.contains("its", case=False)),
    "AvgTimePerBatch"
] /= 2

problems2 = df_time["Problem"].unique()
fig, axes = plt.subplots(1, len(problems2), figsize=(W, W / 2), squeeze=False)
axes = axes[0]

for ax, problem in zip(axes, problems2):
    sub = df_time[df_time["Problem"] == problem].sort_values("Algorithm", ascending=False)
    algos = sub["Algorithm"].tolist()
    times = sub["AvgTimePerBatch"].to_numpy()
    y = np.arange(len(algos))
    bar_colors = [algorithm_colors.get(a, (0.5, 0.5, 0.5)) for a in algos]

    ax.barh(y, times, color=bar_colors, alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(algos, fontsize=8)
    ax.set_xlabel("Time per Batch (s)", fontsize=8)
    ax.set_title(problem)
    ax.grid(axis="x", linestyle="--", alpha=0.7)
    #despine
    sns.despine(ax=ax, left=True, bottom=False)
    #Y-axis ticks only on first subplot
    if ax == axes[0]:
        ax.set_yticks(y)
        ax.set_yticklabels(algos, fontsize=8)
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.tick_params(axis='y', length=0)

plt.savefig(os.path.join(figure_path, "benchmark_avg_time_per_batch.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(figure_path, "benchmark_avg_time_per_batch.pgf"), bbox_inches="tight")
plt.tight_layout()
plt.show()


In [ ]:

from search.objective_generators import _cost_parallel_sa, _cost_pso, _cost_pgd
import os
import torch

# =========================
# CONFIG
# =========================
measure_time = True  # set False to skip the timing benchmark
num_batches = 5  # measured batches (excluding first, plus one extra at end)

# Warm-up the model once to initialize kernels
batch_warmup = next(iter(test_loader_transformed))[0].to(device)
with torch.no_grad():
    _ = model(batch_warmup)
print("Model warm-up done.\n")

# =========================
# BENCHMARK 1: Eval count / cost check
# =========================
print("=== Eval Count / Budget Checking ===")
for budget in budgets:
    print(f"\n--- Budget: {budget} ---")
    for algo in all_algos:
            if algo == "its" or algo in ["shgo_individual", "wcd_lat_ind"]:
                continue
            if algo in complex_algos:
                continue

            # Load params
            algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
            param_path = os.path.join(algo_dir, "best.yml")
            if os.path.exists(param_path):
                stored_params = load_params(param_path)
            else:
                stored_params = get_default_params(algo, budget)

            # Compute cost
            if algo == "shgo":
                cost = _cost_shgo(
                    stored_params["shgo_initial_samples"],
                    stored_params["shgo_local_runs"],
                    stored_params["shgo_local_steps"],
                    stored_params.get("grad_weight", 2),
                )
            elif algo == "parallel_sa":
                cost = _cost_parallel_sa(
                    stored_params["psa_parallel_runs"],
                    stored_params["psa_max_iterations"],
                )
            elif algo == "pso":
                cost = _cost_pso(
                    stored_params["pso_swarm_size"],
                    stored_params["pso_steps"],
                )
            elif algo == "cd":
                cost = None
            elif algo == "wcd":
                cost = None
            elif algo in {"pgd", "pgd_restart", "pgd_window"}:
                cost = _cost_pgd(
                    stored_params["pgd_parallel_runs"],
                    stored_params["pgd_max_iterations"],
                    stored_params.get("grad_weight", 2),
                )
            elif algo == "random_search":
                cost = budget
            else:
                cost = None

            # Build a fresh SinglePass energy problem to count model evals
            model_counter = EvalCounterWrapper(model)
            energy_counter_confidence = SinglePassConfidence(model_counter, EnergyConfidence(), index=None)
            problem_energy_counter = TransformationProblem(
                energy_counter_confidence, transform_seq, consolidate_method="consolidate_simple"
            )
            problem_energy_counter.max_batch_size = dataset_info.batch_size_search

            # Run a single batch to count evaluations
            batch = next(iter(test_loader_transformed))
            model_counter.reset()
            search_obj = build_search_algorithm(algo, stored_params, problem=problems[0], budget=budget,
                                                model=model_counter)
            search_obj.optimize(problem_energy_counter, x=batch[0].to(device))

            print(f"Algorithm: {algo}")
            print(f"  Computed cost: {cost}")
            print(f"  Budget: {budget}")
            print(f"  Eval count per sample: {model_counter.eval_count / len(batch[0]):.2f}")





In [ ]:
print("\n=== Benchmark Summary ===")
for algo, probs in benchmark_results.items():
    print(f"\nAlgorithm: {algo}")
    for prob_name, metrics in probs.items():
        print(f"  {prob_name}: Time {metrics['avg_time_per_batch']:.4f}s")

Loads and plots resutls across all datasets for paper.

In [ ]:
import os
from pathlib import Path

datasets = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
budget = 60

default_architecture_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "coil100": "coil_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

# locate project root (same logic you use elsewhere)
path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    parent = os.path.dirname(current_path)
    if parent == current_path:
        break
    current_path = parent

base_path = Path(current_path) / "experiment_files" / "search_results"
print("base_path:", base_path)
print("base_path exists:", base_path.exists())


def _safe_listdir(p: Path):
    try:
        return sorted([x.name for x in p.iterdir()])
    except Exception as e:
        return [f"<error: {e}>"]


def _peek_json_keys(p: Path, max_keys=20):
    try:
        with p.open("r", encoding="utf-8") as f:
            obj = json.load(f)
        keys = list(obj.keys())
        return keys[:max_keys]
    except Exception as e:
        return [f"<error: {e}>"]


missing_roots = 0
found_any_budget_dir = 0
from data.get_dataset import get_dataset_info

for ds in datasets:
    info = get_dataset_info(ds)
    transform_name = info.transform_seq_name
    arch = default_architecture_mapping[ds]

    dataset_root = base_path / ds / transform_name / arch
    print("\n=== Dataset ===")
    print("dataset:", ds)
    print("transform_name:", transform_name)
    print("arch:", arch)
    print("dataset_root:", dataset_root)
    print("dataset_root exists:", dataset_root.exists())

    if not dataset_root.exists():
        missing_roots += 1
        continue

    algo_dirs = [p for p in dataset_root.iterdir() if p.is_dir()]
    print("algo dirs:", [p.name for p in algo_dirs])

    for algo_dir in algo_dirs:
        budget_dir = algo_dir / f"budget_{budget}"
        if not budget_dir.exists():
            continue

        found_any_budget_dir += 1
        json_files = sorted(list(budget_dir.glob("*.json")))
        print("\n  algo:", algo_dir.name, "| budget_dir:", budget_dir)
        print("  json files:", [p.name for p in json_files][:20])

        # peek first json file keys (to verify accuracy key names)
        if json_files:
            print("  peek keys:", _peek_json_keys(json_files[0]))

print("\n=== Summary ===")
print("missing dataset_roots:", missing_roots, "/", len(datasets))
print("found_any_budget_dir:", found_any_budget_dir)


In [ ]:

from collections import defaultdict

# ===== config =====
datasets = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
budget = 60
problem_names = ["logit_energy", "knn_per_class", "learned_energy"]

default_architecture_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "coil100": "coil_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

# ===== variant groups (dataset-dependent) =====
BASE_VARIANT_GROUPS = {
    "pgd": ["pgd"],
    "shgo": ["shgo"],
    "random_search": ["random_search"],
    "wcd_lattice": ["wcd_lattice"],
    "cd": ["cd"],
    "cd_multi_cyclus": ["cd_multi_cyclus"],
    "wcd": ["wcd"],
    "its": ["its"],
    "pso": ["pso"],
    "parallel_sa": ["parallel_sa"],
}

MODELNET_VARIANT_GROUPS = {
    "pgd": ["pgd"],
    "shgo": ["shgo_vector"],
    "random_search": ["random_search"],
    "wcd_lattice": ["wcd_lat_ind"],
    "cd": ["cd"],
    "cd_multi_cyclus": ["cd_multi_cyclus"],
    "wcd": ["wcd"],
    "its": ["its"],
    "pso": ["pso"],
    "parallel_sa": ["parallel_sa"],
}

TU_VARIANT_GROUPS = {"random_search": ["random_search"],
                     "wcd_lattice": ["wcd_lat_ind"],
                     "cd": ["cd"],
                     "cd_multi_cyclus": ["cd_multi_cyclus"],
                     "wcd": ["wcd"],
                     "its": ["its"],
                     "pso": ["pso"],
                     "parallel_sa": ["parallel_sa"]
                     }


def get_variant_groups(dataset: str):
    if dataset == "modelnet10":
        return MODELNET_VARIANT_GROUPS
    if dataset == "tu_berlin":
        return TU_VARIANT_GROUPS
    return BASE_VARIANT_GROUPS


# Pretty names (optional: extend)

DISPLAY_PROB = {
    "logit_energy": "Logit Energy",
    "knn_per_class": "PC-kNN",
    "learned_energy": "Learned Energy",
}

In [ ]:


import json
import os

results_dict = {}
budget = 60

for ds in datasets:
    variant_groups = get_variant_groups(ds)
    list_results = []
    for base_algo, variants in variant_groups.items():
        print(base_algo)
        for v in variants:
            dsinfo = get_dataset_info(ds)
            brd = os.path.join(
                current_path, "experiment_files", "search_results",
                str(ds), dsinfo.transform_seq_name, default_architecture_mapping[ds],
            )
            algo_dir = os.path.join(brd, v, f"budget_{budget}")

            for method_name in problem_names:
                eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
                if os.path.exists(eval_path):
                    with open(eval_path, "r") as f:
                        metrics = json.load(f)
                list_results.append({
                    "Algorithm": base_algo,
                    "Problem": method_name,
                    "metrics": metrics
                })
    results_dict[ds] = list_results






In [ ]:



def analysis_to_df(analysis):
    """
    Convert the nested 'analysis' dict from analyze_run_results
    into a long DataFrame suitable for plotting.
    """
    rows = []
    for problem, pdata in analysis.items():
        for algo, stats in pdata["algorithms"].items():
            rows.append({
                "Problem": problem,
                "Algorithm": algo,
                "accuracy": stats.get("accuracy"),
                "accuracy_se": stats.get("accuracy_se"),
                "mean_relative_error": stats.get("mean_relative_error"),
                "se_relative_error": stats.get("se_relative_error"),
                "std_relative_error": stats.get("std_relative_error"),
                "mean_frobenius": stats.get("mean_frobenius"),
                "se_frobenius": stats.get("se_frobenius"),
                "std_frobenius": stats.get("std_frobenius"),
                "num_runs": stats.get("num_runs"),
                "num_datapoints": pdata.get("num_datapoints")
            })
    df = pd.DataFrame(rows)
    return df


In [ ]:
analyzed_results = defaultdict(lambda: defaultdict(dict))
for ds in datasets:
    print(ds)
    result_for_ds = results_dict.get(ds, {})
    analyzed_results[ds] = analyze_run_results(result_for_ds)
    analyzed_results[ds] = analysis_to_df(analyzed_results[ds])



In [ ]:
import pandas as pd


def aggregate_with_se(df):
    """
    Groups by Algorithm and averages metrics, while correctly
    propagating Standard Error (SE).
    """
    # Define which columns are means and which are their corresponding SEs
    metric_pairs = {
        'accuracy': 'accuracy_se',
        'mean_relative_error': 'se_relative_error',
        'mean_frobenius': 'se_frobenius'
    }

    # 1. Group by Algorithm
    grouped = df.groupby("Algorithm")
    N = grouped["Problem"].transform("count")  # Number of problems per algorithm

    # 2. Standard Average for the mean metrics
    agg_dict = {m: "mean" for m in metric_pairs.keys()}
    # Keep track of how many problems were averaged
    agg_dict["Problem"] = "count"

    res_df = grouped.agg(agg_dict).rename(columns={"Problem": "n_problems"})

    # 3. Propagate SE: sqrt(sum(se^2)) / N
    for mean_col, se_col in metric_pairs.items():
        if se_col in df.columns:
            # Calculate sum of squares of SEs within each group
            se_sq_sum = df.groupby("Algorithm")[se_col].apply(lambda x: np.sum(x ** 2))
            # Apply the formula: sqrt(sum) / N
            counts = df.groupby("Algorithm").size()
            res_df[se_col] = np.sqrt(se_sq_sum) / counts

    return res_df.reset_index()


# --- Execution ---

processed_dfs = []

for ds_name, df in analyzed_results.items():
    # Perform the averaging over problems for this dataset
    ds_avg = aggregate_with_se(df)

    # Add the dataset name as a column for the final merge
    ds_avg["Dataset"] = ds_name
    processed_dfs.append(ds_avg)

# Merge everything into one master DataFrame
final_results = pd.concat(processed_dfs, ignore_index=True)

# Reorder columns for readability
cols = ["Dataset", "Algorithm", "n_problems"] + \
       [c for c in final_results.columns if c not in ["Dataset", "Algorithm", "n_problems"]]
final_results = final_results[cols]

print(final_results)

In [ ]:
#caluate max accuracy se and max mean relative error se.
final_results["accuracy_se"].max()

In [ ]:
final_results["se_relative_error"].max()

In [ ]:
import pandas as pd
import numpy as np


def fmt(val, se, is_percent=False, include_se=True):
    """
    Formats numbers:
    Accuracy: Value \pm .Unc (1 decimal, scaled by 1.96, rounded UP)
    Error: Value (2 decimals for extra precision, no SE)
    """
    if is_percent:
        val *= 100
        se *= 100

    if include_se:
        # Scale and Round UP to 1 decimal place for Accuracy SE
        se_scaled = 1.96 * se
        se_rounded = np.ceil(se_scaled * 10) / 10
        v_str = f"{val:.1f}"
        s_str = f"{se_rounded:.1f}".replace("0.", ".")
        return f"${v_str} \\pm {s_str}$"
    else:
        # Error metric with 2 decimal places for 'extra position'
        return f"${val:.2f}$"


def get_final_paper_table(df):
    df = df.copy()

    # 1. Format metrics
    df['Acc'] = df.apply(lambda x: fmt(x['accuracy'], x['accuracy_se'], True, True), axis=1)
    df['Err'] = df.apply(lambda x: fmt(x['mean_relative_error'], x['se_relative_error'], False, False), axis=1)

    # 2. Pivot: Dataset as top level, Algorithm as index
    pivot = df.pivot(index='Algorithm', columns='Dataset', values=['Acc', 'Err'])

    # 3. Reorder: Dataset Name -> [Acc, Err]
    pivot = pivot.reorder_levels([1, 0], axis=1).sort_index(axis=1)

    datasets = sorted(df['Dataset'].unique())
    num_ds = len(datasets)

    # Header 1: Datasets
    header_ds = "Dataset & " + " & ".join([f"\\multicolumn{{2}}{{r}}{{{ds}}}" for ds in datasets]) + " \\\\"

    # Header 2: Acc/Err
    header_metrics = " & " + " & ".join(["Acc & Err" for _ in datasets]) + " \\\\"

    # Body rows
    rows = []
    for algo in pivot.index:
        row_str = f"{algo} & " + " & ".join([str(pivot.loc[algo, (ds, m)]) for ds in datasets for m in ['Acc', 'Err']])
        rows.append(row_str + " \\\\")

    # Combine into full tabular
    tabular_header = "\\begin{tabular}{l" + "rr" * num_ds + "}"
    full_table = [
        tabular_header,
        "\\toprule",
        header_ds,
        header_metrics,
        "Algorithm & " + " & " * (num_ds * 2 - 1) + " \\\\",
        "\\midrule",
        "\n".join(rows),
        "\\bottomrule",
        "\\end{tabular}"
    ]

    return "\n".join(full_table)


print(get_final_paper_table(final_results))